# Analiza ekstrakcji formularzy względem oczekiwanych wyników
**Zakres:**  **14 wariantów źródłowych**
(po 7 formatów dla dwóch wniosków), każdy w trzech reprezentacjach
(`fields`, `simple`, `form`). Każdy wariant jest porównywany
z odpowiednim z **dwóch oczekiwanych wyników** znajdujących się w folderze `wyniki`:
**Przedmiot analizy:** (1) jaka jest trafność ekstrakcji względem wzorca, (2) czy zaproponowane przez nas rozwiazanie sprawdza sie tak samo dobrze niezależnie od formatu źródłowego.

## Dopasowywanie pól i NLP
Pola są dopasowywane **jeden do jednego** w trzech krokach: (1) unikatowa nazwa
techniczna (tylko `form`), (2) unikatowa identyczna etykieta w danym zakresie,
(3) Według podobieństwa leksykalnego: rdzenie 5-znakowe i znakowe n-gramy.
W celu dopelnienia powyzszych metod stosujemy również **dopasowanie metoda ręcznej weryfikacji semantycznej**, które pozwala na uwzględnienie kontekstu i znaczenia pól, niezależnie od ich dokładnej etykiety czy nazwy technicznej.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from functools import lru_cache
import csv
import hashlib
import html
import itertools
import json
import math
import random
import re
import statistics
import sys
import unicodedata
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "wyniki" / "s"
OUTPUT_DIR = NOTEBOOK_DIR / "analiza_wyniki"
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
PRIMARY_THRESHOLD = 0.60 
THRESHOLDS = (0.50, 0.60, 0.70, 0.80)
CONTENT = "treść wniosku"
COMMON = "wspólny szablon"
ALL_INPUTS = "wszystkie pola wejściowe"
TABLES = {}
SVG_COUNT = 0
def emit_html(markup):
    if "_capture_html" in globals():
        _capture_html(markup)
    else:
        from IPython.display import HTML, display
        display(HTML(markup))
def fmt(value):
    if value is None:
        return "—"
    if isinstance(value, bool):
        return "tak" if value else "nie"
    if isinstance(value, float):
        return f"{value:.3f}" if math.isfinite(value) else str(value)
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)
def table(name, rows, limit=30, columns=None, title=None):
    rows = list(rows)
    TABLES[name] = rows
    heading = html.escape(title or name)
    if not rows:
        emit_html(f"<h3>{heading}</h3><p>Brak wierszy.</p>")
        return
    columns = columns or list(dict.fromkeys(k for row in rows for k in row))
    head = "".join(f"<th style='background:#eef2f7;padding:3px 6px'>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td style='padding:2px 6px'>{html.escape(fmt(row.get(c)))}</td>" for c in columns) + "</tr>"
        for row in rows[:limit])
    emit_html(f"<h3>{heading}</h3><div style='overflow:auto;max-height:600px'>"
              f"<table style='border-collapse:collapse;font-size:12px' border='1'><thead><tr>{head}</tr></thead>"
              f"<tbody>{body}</tbody></table></div>")
def show_svg(name, markup):
    global SVG_COUNT
    SVG_COUNT += 1
    (OUTPUT_DIR / "plots" / f"{SVG_COUNT:02d}_{name}.svg").write_text(markup, encoding="utf-8")
    emit_html(markup)
PALETTE = ("#2563eb", "#e87924", "#16a34a", "#9333ea", "#dc2626")
def bar_chart(name, title, labels, series, maximum=1.0, note=""):
    row_h = max(34, len(series) * 16 + 12)
    longest = max((len(str(label)) for label in labels), default=10)
    plot_w = 520
    left = int(min(560, max(200, 6.4 * longest + 28)))
    width = max(left + plot_w + 90, 760)
    height = 90 + len(labels) * row_h
    svg = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' "
           f"viewBox='0 0 {width} {height}' role='img' aria-label='{html.escape(title)}'>",
           "<rect width='100%' height='100%' fill='white'/>",
           f"<text x='12' y='24' font-size='17' font-family='sans-serif'>{html.escape(title)}</text>"]
    if note:
        svg.append(f"<text x='12' y='42' font-size='11' fill='#555' font-family='sans-serif'>{html.escape(note)}</text>")
    for j, (label, _) in enumerate(series):
        svg.append(f"<rect x='{12+j*230}' y='50' width='12' height='12' fill='{PALETTE[j%5]}'/>"
                   f"<text x='{30+j*230}' y='61' font-size='12' font-family='sans-serif'>{html.escape(label)}</text>")
    for i, label in enumerate(labels):
        y = 78 + i * row_h
        svg.append(f"<text x='10' y='{y+12}' font-size='11' font-family='sans-serif'>{html.escape(str(label))}</text>")
        for j, (_, values) in enumerate(series):
            value = values[i]
            yy = y + j * 16
            if value is None:
                svg.append(f"<text x='{left}' y='{yy+11}' font-size='10' fill='#888'>brak danych</text>")
                continue
            length = plot_w * min(max(value / maximum, 0), 1)
            text = f"{value:.3f}" if maximum <= 1.0 else f"{value:.0f}" if float(value).is_integer() else f"{value:.2f}"
            svg.append(f"<rect x='{left}' y='{yy}' width='{length:.1f}' height='13' fill='{PALETTE[j%5]}'/>"
                       f"<text x='{left+length+5:.1f}' y='{yy+11}' font-size='10' font-family='sans-serif'>{text}</text>")
    svg.append("</svg>")
    show_svg(name, "".join(svg))
def heatmap(name, title, row_labels, col_labels, values, texts=None, scale=1.0, note=""):
    longest = max((len(str(label)) for label in row_labels), default=10)
    left = int(min(520, max(160, 6.4 * longest + 28)))
    top, cw, ch = 150, 64, 34
    width = max(left + cw * len(col_labels) + 30, 760)
    height = top + ch * len(row_labels) + 30
    svg = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' "
           f"viewBox='0 0 {width} {height}' role='img' aria-label='{html.escape(title)}'>",
           "<rect width='100%' height='100%' fill='white'/>",
           f"<text x='12' y='24' font-size='16' font-family='sans-serif'>{html.escape(title)}</text>",
           f"<text x='12' y='44' font-size='11' fill='#555' font-family='sans-serif'>{html.escape(note)}</text>"]
    for j, label in enumerate(col_labels):
        x = left + j * cw + cw / 2
        svg.append(f"<text transform='translate({x},{top-10}) rotate(-45)' font-size='11' "
                   f"font-family='sans-serif'>{html.escape(str(label))}</text>")
    for i, label in enumerate(row_labels):
        y = top + i * ch
        svg.append(f"<text x='10' y='{y+21}' font-size='11' font-family='sans-serif'>{html.escape(str(label))}</text>")
        for j in range(len(col_labels)):
            v = values[i][j]
            t = None if v is None else min(1.0, max(0.0, v / scale))
            color = "#eeeeee" if t is None else f"rgb({int(240-195*t)},{int(246-126*t)},{int(255-55*t)})"
            text = texts[i][j] if texts else ("—" if v is None else f"{v:.2f}")
            dark = t is not None and t > 0.62
            svg.append(f"<rect x='{left+j*cw}' y='{y}' width='{cw-2}' height='{ch-2}' fill='{color}'/>"
                       f"<text x='{left+j*cw+cw/2}' y='{y+21}' text-anchor='middle' font-size='12' "
                       f"font-family='sans-serif' fill='{'white' if dark else 'black'}'>{html.escape(text)}</text>")
    svg.append("</svg>")
    show_svg(name, "".join(svg))
def clean_text(value):
    return html.unescape(re.sub(r"<[^>]*>", " ", str(value or ""))).strip()
def norm(value):
    text = clean_text(value).casefold().replace("ł", "l")
    text = "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))
    return " ".join(re.findall(r"[a-z0-9]+", text))
def ratio(a, b):
    return a / b if b else None
def mean(values):
    values = [v for v in values if v is not None]
    return statistics.mean(values) if values else None
def f1_score(p, r):
    if p is None or r is None:
        return None
    return 2 * p * r / (p + r) if p + r else 0.0
print("Python:", sys.version.split()[0])
print("Dane wejściowe:", DATA_DIR)
print("Wzorce:", NOTEBOOK_DIR / "wyniki")
print("Wyniki zapisują się w:", OUTPUT_DIR)

## 1. Wczytanie danych, parowanie i kontrola kompletności

In [ ]:
FILE_RE = re.compile(r"^(?P<doc>.+)\.(?P<fmt>[^.]+)\.(?P<rep>fields|simple|form)\.json$", re.I)
import pandas as pd
records = []
for path in sorted(DATA_DIR.glob("*.json")):
    m = FILE_RE.match(path.name)
    if m:
        fmt_ = m["fmt"].casefold()
        records.append({"name": path.name, "document": m["doc"], "format": fmt_, "representation": m["rep"].casefold(),
                        "source_id": norm(m["doc"]) + "." + fmt_,
                        "data": json.loads(path.read_text(encoding="utf-8-sig"))})
references = {}
for path in sorted((NOTEBOOK_DIR / "wyniki").glob("konfiguracja_form_*.json")):
    obj = json.loads(path.read_text(encoding="utf-8-sig"))
    references[norm(obj["title"])] = {"file": path.name, "data": obj}
by_source = defaultdict(dict)
for rec in records:
    by_source[rec["source_id"]][rec["representation"]] = rec
PIPELINE_NAMES = {"pdf,vlm": "tekst z PDF + VLM", "ocr:tesseract,vlm": "OCR (Tesseract) + VLM"}
source_meta = {}
for source_id, reps in sorted(by_source.items()):
    fields_data = reps["fields"]["data"]
    pipeline = ",".join(fields_data.get("text_source", []))
    source_meta[source_id] = {"source_id": source_id, "document": reps["form"]["document"],
        "ref_key": norm(reps["form"]["document"]), "format": reps["form"]["format"],
        "label": reps["form"]["format"].upper(), "pipeline_raw": pipeline,
        "pipeline": PIPELINE_NAMES.get(pipeline, pipeline),
        "vlm_model": fields_data.get("vlm", {}).get("model"), "pages": fields_data.get("pages")}
DOC_SHORT = {norm("Wniosek o stwierdzenie nadpłaty podatku"): "Nadpłata podatku",
             norm("Wniosek o ustalenie numeru porządkowego nieruchomości"): "Numer porządkowy nieruchomości"}
for meta in source_meta.values():
    meta["doc_short"] = DOC_SHORT.get(meta["ref_key"], meta["document"])
files_df = pd.DataFrame(records).drop(columns=["data", "source_id"])
sources_df = pd.DataFrame(source_meta.values())[["doc_short", "format", "vlm_model", "pages", "ref_key"]]
sources_df = sources_df.assign(wzorzec=sources_df["ref_key"].map(lambda k: references[k]["file"])).drop(columns="ref_key")
TABLES["wykaz_plikow"] = files_df.to_dict("records")
TABLES["parowanie_zrodel"] = sources_df.to_dict("records")
display(files_df.head(12), sources_df)
print(f"Pliki JSON: {len(files_df)} | warianty: {len(by_source)} | wzorce: {len(references)}")

## 2. Normalizacja pól do wspólnej postaci

In [ ]:
DECORATIVE = {"SECTION", "TEXT_BLOCK", "INFO_SECTION", "CONSENT_SECTION"}
RAW_TO_FORM = {"string": "TEXT_FIELD", "textarea": "TEXTAREA", "number": "NUMBER", "address": "TEXT_FIELD",
               "postal_code": "TEXT_FIELD", "email": "TEXT_FIELD", "phone": "TEXT_FIELD", "pesel": "TEXT_FIELD",
               "nip": "TEXT_FIELD", "regon": "TEXT_FIELD", "iban": "TEXT_FIELD", "date": "DATEPICKER",
               "file": "FILE_UPLOADER", "radio": "RADIO", "checkbox": "CHECKBOX", "signature": "SIGNATURE"}
CONTENT_ROLES = {"subject", "attachment", "statement"}
schema_notes = []
def category_of(label, form_type):
    text = norm(label)
    if form_type in DECORATIVE:
        return "strukturalne"
    if form_type == "FILE_UPLOADER" or re.search(r"\b(plik|zalacznik|dokument|potwierdzeni)", text):
        return "załączniki"
    if form_type == "SIGNATURE" or "podpis" in text:
        return "podpis"
    if re.search(r"telefon|e mail|email", text):
        return "kontakt"
    if re.search(r"rachun|iban|kwot|oplat|podat|nadplat|przelew", text):
        return "kwoty, podatki, płatności"
    if form_type == "DATEPICKER" or re.search(r"\bdata\b", text):
        return "data"
    if re.search(r"pesel|regon|\bnip\b|krs|imie|nazwisk|\bnazwa\b|wybierz jako kto|w czyim imieniu", text):
        return "tożsamość i podmiot"
    if re.search(r"adres|ulic|miejscow|kod poczt|wojewodz|teryt|gmin|powiat|kraj|budyn|lokal|dzialk|obreb", text):
        return "adres i nieruchomość"
    if form_type in ("RADIO", "SELECT", "CHECKBOX"):
        return "wybór"
    if re.search(r"uzasadnien|opis", text):
        return "uzasadnienie i opis"
    return "pozostałe"
def tab_scope(tab):
    text = norm(tab)
    return COMMON if text == "dane" or re.search(r"kontakt|zgod|koncow", text) else CONTENT
def option_names(options):
    if not options:
        return []
    if not isinstance(options, list):
        raise ValueError("Opcje muszą być listą")
    return [clean_text(o.get("name", o.get("label", o.get("id", "")))) if isinstance(o, dict) else clean_text(o)
            for o in options]
def rules_text(value):
    return json.dumps(value or [], ensure_ascii=False, sort_keys=True)
def form_rows(obj, meta):
    rows = []
    def walk(questions, tab, section, address):
        for index, question in enumerate(questions or []):
            field = question["field"]
            label = clean_text(field.get("title"))
            form_type = field.get("type", "")
            validation = field.get("validation", [])
            if validation in ({}, "", None):
                if validation == {}:
                    schema_notes.append({"plik": meta["name"], "pole": label, "problem": "validation = {}",
                                         "normalizacja": "potraktowano jako brak reguł walidacji"})
                validation = []
            states = field.get("states", [])
            if states in ({}, "", None):
                states = []
            rows.append({**meta, "uid": f"{meta['name']}:{address}/{index}", "label": label,
                         "form_type": form_type, "raw_type": form_type, "section": section, "tab": tab,
                         "scope": tab_scope(tab), "is_input": form_type not in DECORATIVE,
                         "tech_name": field.get("techName", ""), "role": "", "detector": None, "confidence": None,
                         "required": "REQUIRED" in validation, "rules": rules_text(validation),
                         "states": rules_text(states), "options": option_names(field.get("options")),
                         "read_only": bool(field.get("readOnly", False)),
                         "default": json.dumps(field.get("defaultValue"), ensure_ascii=False),
                         "category": category_of(label, form_type)})
            walk(field.get("repeatingSectionGrid"), tab,
                 " / ".join(x for x in (section, label) if x), f"{address}/{index}")
    for i, tab in enumerate(obj["tabs"]):
        walk(tab["questions"], tab["title"], "", f"tab{i}")
    return rows
def raw_rows(obj, meta, simple=False):
    items = obj["pola"] if simple else obj["fields"]
    rows = []
    for i, item in enumerate(items):
        label = clean_text(item.get("Opis pola" if simple else "label"))
        raw_type = item.get("Rodzaj szczegółowy") or item.get("Rodzaj pola") if simple else item.get("type")
        form_type = RAW_TO_FORM.get(raw_type, "TEXT_FIELD")
        role = item.get("Rola" if simple else "role", "")
        rows.append({**meta, "uid": f"{meta['name']}:{i}", "label": label, "form_type": form_type,
                     "raw_type": raw_type, "section": clean_text(item.get("Sekcja" if simple else "section")),
                     "tab": "", "scope": CONTENT if role in CONTENT_ROLES else "poza treścią wniosku",
                     "is_input": True, "tech_name": "", "role": role,
                     "detector": None if simple else item.get("detector"),
                     "confidence": None if simple else item.get("confidence"),
                     "required": item.get("Wymagane" if simple else "required"),
                     "rules": None, "states": None, "options": option_names(item.get("Opcje" if simple else "options")),
                     "read_only": None, "default": None, "category": category_of(label, form_type)})
    return rows
per_record, all_rows, reference_rows, quality = {}, [], {}, []
for rec in records:
    m = source_meta[rec["source_id"]]
    meta = {"name": rec["name"], "source_id": rec["source_id"], "document": m["doc_short"], "format": m["format"],
            "representation": rec["representation"], "pipeline": m["pipeline"], "ref_key": m["ref_key"]}
    if rec["representation"] == "form":
        rows = form_rows(rec["data"], meta)
    else:
        rows = raw_rows(rec["data"], meta, simple=rec["representation"] == "simple")
    per_record[rec["name"]] = rows
    all_rows.extend(rows)
for key, ref in references.items():
    meta = {"name": "wzorzec:" + ref["file"], "source_id": "wzorzec:" + key, "document": DOC_SHORT[key],
            "format": "wzorzec", "representation": "wzorzec", "pipeline": "wzorzec", "ref_key": key}
    reference_rows[key] = form_rows(ref["data"], meta)
for source_id, reps in sorted(by_source.items()):
    m = source_meta[source_id]
    form_in = [r for r in per_record[reps["form"]["name"]] if r["is_input"]]
    raw_in = per_record[reps["fields"]["name"]]
    ref_in = [r for r in reference_rows[m["ref_key"]] if r["is_input"]]
    form_data = reps["form"]["data"]
    tabs = [t["title"] for t in form_data["tabs"]]
    ref_tabs = [t["title"] for t in references[m["ref_key"]]["data"]["tabs"]]
    quality.append({"dokument": m["doc_short"], "format": m["format"], "przetwarzanie_tekstu": m["pipeline"],
        "pola_fields": len(raw_in), "pola_wejsciowe_form": len(form_in), "pola_wejsciowe_wzorca": len(ref_in),
        "wszystkie_komponenty_form": len(per_record[reps["form"]["name"]]),
        "zakladki_form": len(tabs), "zakladki_wzorca": len(ref_tabs),
        "tytul_form": form_data["title"], "tytul_fields": reps["fields"]["data"]["title"],
        "tytul_form_zgodny": norm(form_data["title"]) == m["ref_key"],
        "tytul_fields_zgodny": norm(reps["fields"]["data"]["title"]) == m["ref_key"],
        "simple_rowne_fields_simple": reps["fields"]["data"].get("simple") == reps["simple"]["data"].get("pola"),
        "puste_etykiety": sum(not norm(r["label"]) for r in form_in + raw_in)})
table("jakosc_i_struktura", quality, limit=14, title="Struktura i jakość odczytu (14 wariantów)")
table("uwagi_schematu", schema_notes, limit=10, title="Normalizacje schematu")
print("Znormalizowane wiersze:", len(all_rows), "| wiersze wzorców:", sum(len(v) for v in reference_rows.values()))
print("simple == fields.simple:", sum(q["simple_rowne_fields_simple"] for q in quality), "/", len(quality))
print("Pola z REQUIRED w naszych formularzach:", sum(r["required"] for rec in records if rec["representation"] == "form"
                                                      for r in per_record[rec["name"]]))

## 3. Ocena podobieństwa etykiet
**Podobieństwo etykiet** = Słowa są skracane do rdzeni 5-znakowych, optymalnych dla polskiej fleksji (* np.podatku/podatek → podat*).

In [ ]:
STOPWORDS = set("i w z ze na do o od po za dla lub albo a czy jest sie nie jesli co oraz ktory ktorego "
                "ktore ktora ktorej jako".split())
def stems(text):
    return [t[:5] for t in norm(text).split() if t not in STOPWORDS]
@lru_cache(maxsize=None)
def features(text):
    counts = Counter("w:" + s for s in stems(text))
    padded = " " + norm(text) + " "
    for n in (3, 4):
        counts.update("c:" + padded[i:i + n] for i in range(max(0, len(padded) - n + 1)))
    return counts
corpus = sorted({norm(r[k]) for r in all_rows + [x for v in reference_rows.values() for x in v]
                 for k in ("label", "section") if norm(r[k])})
doc_freq = Counter(f for text in corpus for f in features(text))
IDF = {f: math.log((1 + len(corpus)) / (1 + c)) + 1 for f, c in doc_freq.items()}
DEFAULT_IDF = math.log(1 + len(corpus)) + 1
@lru_cache(maxsize=None)
def vector(text):
    values = {f: (1 + math.log(c)) * IDF.get(f, DEFAULT_IDF) for f, c in features(text).items()}
    length = math.sqrt(sum(v * v for v in values.values()))
    return {f: v / length for f, v in values.items()} if length else {}
@lru_cache(maxsize=None)
def lexical_similarity(a, b):
    na, nb = norm(a), norm(b)
    if not na or not nb:
        return 0.0
    if na == nb:
        return 1.0
    va, vb = vector(na), vector(nb)
    cosine = sum(v * vb.get(f, 0.0) for f, v in va.items())
    sa, sb = set(stems(na)), set(stems(nb))
    weight = lambda s: sum(IDF.get("w:" + x, DEFAULT_IDF) for x in s)
    overlap = weight(sa & sb) / min(weight(sa), weight(sb)) if sa and sb else 0.0
    return 0.5 * cosine + 0.5 * overlap
def similarity(a, b):
    label = lexical_similarity(a["label"], b["label"])
    if label and a["section"] and b["section"]:
        return 0.85 * label + 0.15 * lexical_similarity(a["section"], b["section"])
    return label
def hungarian(cost):
    n, m = len(cost), len(cost[0])
    u, v, p, way = [0.0] * (n + 1), [0.0] * (m + 1), [0] * (m + 1), [0] * (m + 1)
    for i in range(1, n + 1):
        p[0], j0 = i, 0
        minv, used = [float("inf")] * (m + 1), [False] * (m + 1)
        while True:
            used[j0] = True
            i0, delta, j1 = p[j0], float("inf"), 0
            for j in range(1, m + 1):
                if not used[j]:
                    cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j], way[j] = cur, j0
                    if minv[j] < delta:
                        delta, j1 = minv[j], j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
    assignment = [-1] * n
    for j in range(1, m + 1):
        if p[j]:
            assignment[p[j] - 1] = j - 1
    return assignment
def assign(scores, threshold):
    if not scores or not scores[0]:
        return []
    n, m = len(scores), len(scores[0])
    bonus = min(n, m) + 1                       # najpierw maksymalna liczba par, potem suma podobieństw
    cost = [[-(bonus + s) if s >= threshold else 1e6 for s in row] + [0.0] * n for row in scores]
    return [(i, j, scores[i][j]) for i, j in enumerate(hungarian(cost)) if j < m and scores[i][j] >= threshold]
rng = random.Random(2026)
for n, m in itertools.product(range(1, 5), repeat=2):
    for _ in range(15):
        grid = [[rng.choice((0.0, 0.55, 0.59, 0.6, 0.7, 0.9, 1.0)) for _ in range(m)] for _ in range(n)]
        got = assign(grid, 0.6)
        best = (0, 0.0)
        for choice in itertools.product(range(-1, m), repeat=n):
            real = [j for j in choice if j >= 0]
            pairs = [(i, j) for i, j in enumerate(choice) if j >= 0]
            if len(real) == len(set(real)) and all(grid[i][j] >= 0.6 for i, j in pairs):
                best = max(best, (len(pairs), sum(grid[i][j] for i, j in pairs)))
        assert (len(got), round(sum(s for _, _, s in got), 9)) == (best[0], round(best[1], 9))
assert norm("  Łódź, <b>Imię</b> ") == "lodz imie"
assert lexical_similarity("Ulica", "ulica ") == 1.0
print("Test algorytmu przypisania: 240 losowych przypadków zgodnych z pełnym przeszukaniem.")
print("Unikatowe teksty NLP:", len(corpus), "| cechy:", len(IDF))
examples = [("Kwota", "Kwota nadpłaty"), ("Rodzaj podatku", "Podatek, którego dotyczy nadpłata"),
            ("Sposób rozliczenia nadpłaty", "Wybierz, co mamy zrobić z nadpłatą"),
            ("Numer rachunku (jeśli zwrócony przelewem)", "Numer rachunku"), ("Obręb", "Numer ewidencyjny działki")]
table("przyklady_podobienstwa", [{"etykieta_A": a, "etykieta_B": b, "podobienstwo": lexical_similarity(a, b),
                                  "powyzej_progu_0_60": lexical_similarity(a, b) >= PRIMARY_THRESHOLD}
                                 for a, b in examples], title="Przykłady podobieństwa leksykalnego (ilustracja działania NLP)")

## 4. Obliczenie zgodności ze wzorcami
Dla każdego z 14 wariantów porównujemy:
* `form` ze wzorcem w trzech zakresach: wszystkie pola wejściowe, wspólny szablon, treść wniosku;
* `fields` i `simple` ze wzorcem wyłącznie w zakresie treści wniosku.
Atrybuty dopasowanych pól: **typ** (typ FormDefinition), **etykieta** (po normalizacji),
**opcje** (zbiór nazw), a dla `form` także **tryb tylko do odczytu**, **wartość domyślna**
i **warunki stanów**. „Ścisła zgodność treści” wymaga zgodności wszystkich tych atrybutów.
**Ręczna weryfikacja semantyczna.** Dla zakresu treści wniosku na potrzeby tej analizy proponujemy jawną tabelę
odpowiedników (zmienna `MANUAL` poniżej). Dzieli ona dopasowania na *pewne* i *niepewne*.

In [ ]:
def evaluate(pred, exp, threshold=PRIMARY_THRESHOLD, same_schema=False, scoped=True):
    matches, used_p, used_e = [], set(), set()
    compatible = (lambda a, b: a["scope"] == b["scope"]) if scoped else (lambda a, b: True)
    def anchor(key, reason):
        pi, ei = defaultdict(list), defaultdict(list)
        for i, r in enumerate(pred):
            if i not in used_p and key(r):
                pi[key(r)].append(i)
        for j, r in enumerate(exp):
            if j not in used_e and key(r):
                ei[key(r)].append(j)
        for k in sorted(pi.keys() & ei.keys(), key=str):
            if len(pi[k]) == len(ei[k]) == 1 and compatible(pred[pi[k][0]], exp[ei[k][0]]):
                i, j = pi[k][0], ei[k][0]
                matches.append((i, j, similarity(pred[i], exp[j]), reason))
                used_p.add(i)
                used_e.add(j)
    if same_schema:
        anchor(lambda r: norm(r["tech_name"]), "unikatowa nazwa techniczna")
    anchor(lambda r: (norm(r["label"]), r["scope"] if scoped else "") if norm(r["label"]) else None,
           "unikatowa identyczna etykieta")
    pi = [i for i in range(len(pred)) if i not in used_p]
    ej = [j for j in range(len(exp)) if j not in used_e]
    scores = [[similarity(pred[i], exp[j]) if compatible(pred[i], exp[j]) else 0.0 for j in ej] for i in pi]
    for a, b, score in assign(scores, threshold):
        matches.append((pi[a], ej[b], score, "podobieństwo NLP"))
        used_p.add(pi[a])
        used_e.add(ej[b])
    extra = [i for i in range(len(pred)) if i not in used_p]
    missing = [j for j in range(len(exp)) if j not in used_e]
    return summarize(pred, exp, matches, same_schema), matches, extra, missing
def pair_flags(a, b, same_schema):
    flags = {"etykieta": norm(a["label"]) == norm(b["label"]), "typ": a["form_type"] == b["form_type"],
             "opcje": Counter(norm(x) for x in a["options"]) == Counter(norm(x) for x in b["options"])}
    if same_schema:
        flags.update({"tryb_odczytu": a["read_only"] == b["read_only"], "wartosc_domyslna": a["default"] == b["default"],
                      "stany": a["states"] == b["states"]})
    flags["scisla"] = all(flags.values())
    return flags
def summarize(pred, exp, matches, same_schema):
    tp = len(matches)
    p, r = ratio(tp, len(pred)), ratio(tp, len(exp))
    flags = [pair_flags(pred[i], exp[j], same_schema) for i, j, _, _ in matches]
    res = {"tp": tp, "fp": len(pred) - tp, "fn": len(exp) - tp, "predykcje": len(pred), "oczekiwane": len(exp),
           "precyzja": p, "czulosc": r, "f1": f1_score(p, r),
           "srednie_podobienstwo": mean(s for _, _, s, _ in matches),
           "min_podobienstwo": min((s for _, _, s, _ in matches), default=None)}
    for key in ("etykieta", "typ", "opcje", "scisla") + (("tryb_odczytu", "wartosc_domyslna", "stany") if same_schema else ()):
        res["zgodne_" + key] = sum(f[key] for f in flags)
    return res
MANUAL = {
    norm("Wniosek o stwierdzenie nadpłaty podatku"): {
        "kwota nadplaty": ("Kwota", "pewne"), "rodzaj podatku": ("Podatek, którego dotyczy nadpłata", "pewne"),
        "uzasadnienie": ("Uzasadnienie", "pewne"),
        "sposob rozliczenia nadplaty": ("Wybierz, co mamy zrobić z nadpłatą", "pewne"),
        "numer rachunku": ("Numer rachunku", "pewne"),
        "numer rachunku jesli zwrocony przelewem": ("Numer rachunku", "pewne"),
        "inne zalaczniki": ("Plik deklaracji podatkowej", "pewne"),
        "tytul nadplaty": ("Nazwa podatku", "pewne"),
        "tytul nadplaty jesli zaliczony na poczet": ("Nazwa podatku", "pewne"),
        "tytul nadplaty do zaliczenia": ("Nazwa podatku", "pewne")},
    norm("Wniosek o ustalenie numeru porządkowego nieruchomości"): {
        "miejscowosc": ("Miejscowość", "pewne"), "ulica": ("Ulica", "pewne"),
        "numer ewidencyjny dzialki": ("Numer ewidencyjny działki", "pewne"), "obreb": ("Obręb", "pewne")}}
def manual_evaluate(pred, exp, ref_key):
    mapping, used, pairs = MANUAL[ref_key], set(), []
    for i, row in enumerate(pred):
        target = mapping.get(norm(row["label"]))
        if not target:
            continue
        for j, ref in enumerate(exp):
            if j not in used and norm(ref["label"]) == norm(target[0]):
                used.add(j)
                pairs.append((i, j, target[1]))
                break
    out = {}
    for name, tiers in (("pewne", {"pewne"}), ("pewne+niepewne", {"pewne", "niepewne"})):
        tp = sum(t in tiers for _, _, t in pairs)
        p, r = ratio(tp, len(pred)), ratio(tp, len(exp))
        out[name] = {"tp": tp, "precyzja": p, "czulosc": r, "f1": f1_score(p, r)}
    return out, pairs
EVAL, AUDIT, UNMATCHED, MATCH_STORE, SENS, MANUAL_ROWS = [], [], [], {}, [], []
SCOPE_SETS = {ALL_INPUTS: lambda r: True, COMMON: lambda r: r["scope"] == COMMON, CONTENT: lambda r: r["scope"] == CONTENT}
REP_NAMES = {"form": "form", "fields": "fields", "simple": "simple"}
for source_id, reps in sorted(by_source.items()):
    m = source_meta[source_id]
    ref_inputs = [r for r in reference_rows[m["ref_key"]] if r["is_input"]]
    for rep in ("form", "fields", "simple"):
        rows = per_record[reps[rep]["name"]]
        rows = [r for r in rows if r["is_input"]]
        scopes = list(SCOPE_SETS) if rep == "form" else [CONTENT]
        for scope in scopes:
            pred = [r for r in rows if SCOPE_SETS[scope](r)]
            exp = [r for r in ref_inputs if SCOPE_SETS[scope](r)]
            res, matches, extra, missing = evaluate(pred, exp, same_schema=rep == "form")
            meta = {"dokument": m["doc_short"], "format": m["format"], "przetwarzanie_tekstu": m["pipeline"],
                    "reprezentacja": REP_NAMES[rep], "rep": rep, "zakres": scope, "source_id": source_id}
            EVAL.append({**meta, **res})
            MATCH_STORE[(source_id, rep, scope)] = (pred, exp, matches, extra, missing)
            if scope in (ALL_INPUTS, CONTENT) or rep != "form":
                for i, j, score, reason in matches:
                    a, b = pred[i], exp[j]
                    flags = pair_flags(a, b, rep == "form")
                    AUDIT.append({**meta, "etykieta_wyniku": a["label"], "etykieta_wzorca": b["label"],
                                  "sekcja_wyniku": a["section"], "sekcja_wzorca": b["section"],
                                  "podobienstwo": score, "podstawa": reason, "typ_wyniku": a["form_type"],
                                  "typ_wzorca": b["form_type"], "opcje_wyniku": a["options"], "opcje_wzorca": b["options"],
                                  "kategoria": b["category"], "detektor": a["detector"], "pewnosc": a["confidence"],
                                  **{"zgodne_" + k: v for k, v in flags.items()}})
            for i in extra:
                UNMATCHED.append({**meta, "rodzaj": "w wyniku, brak we wzorcu", "etykieta": pred[i]["label"],
                                  "sekcja": pred[i]["section"], "kategoria": pred[i]["category"],
                                  "rola": pred[i]["role"], "detektor": pred[i]["detector"]})
            for j in missing:
                UNMATCHED.append({**meta, "rodzaj": "we wzorcu, brak w wyniku", "etykieta": exp[j]["label"],
                                  "sekcja": exp[j]["section"], "kategoria": exp[j]["category"], "rola": "", "detektor": None})
            if scope == CONTENT:
                for threshold in THRESHOLDS:
                    s, _, _, _ = evaluate(pred, exp, threshold, rep == "form")
                    SENS.append({**meta, "prog": threshold, **s})
                manual, pairs = manual_evaluate(pred, exp, m["ref_key"])
                for variant, values in manual.items():
                    MANUAL_ROWS.append({**meta, "dopasowanie": "ręczne: " + variant, **values,
                                        "predykcje": len(pred), "oczekiwane": len(exp)})
                MATCH_STORE[(source_id, rep, "ręczne")] = (pred, exp, pairs)
print("Wiersze wyników:", len(EVAL), "| dopasowania:", len(AUDIT), "| niedopasowane:", len(UNMATCHED))
assert all(r["tp"] + r["fp"] == r["predykcje"] and r["tp"] + r["fn"] == r["oczekiwane"] for r in EVAL)
assert all(r["zgodne_scisla"] <= r["tp"] for r in EVAL)
for (source_id, rep, scope), value in MATCH_STORE.items():
    if scope != "ręczne":
        pairs = [(i, j) for i, j, _, _ in value[2]]
        assert len({i for i, _ in pairs}) == len(pairs) == len({j for _, j in pairs})
print("Kontrola ograniczenia jeden do jednego: poprawna dla wszystkich dopasowań.")

## 5. Pytanie 1 — trafność względem wzorca
Tabela poniżej zawiera wszystkie porównania: 14 wariantów × (`form` w 3 zakresach, `fields`
i `simple` w zakresie treści wniosku). TP, FP, FN to liczby pól: dopasowane, nadmiarowe
w naszym wyniku i brakujące względem wzorca. **Wyniki zbiorcze sumują pola wszystkich
wariantów (mikro), a warianty tego samego dokumentu nie są niezależnymi próbkami.**

In [ ]:
def aggregate(rows, keys):
    groups = defaultdict(list)
    for r in rows:
        groups[tuple(r[k] for k in keys)].append(r)
    out = []
    for key, g in sorted(groups.items(), key=lambda x: str(x[0])):
        tp, pr, ex = sum(r["tp"] for r in g), sum(r["predykcje"] for r in g), sum(r["oczekiwane"] for r in g)
        p, rc = ratio(tp, pr), ratio(tp, ex)
        strict = sum(r["zgodne_scisla"] for r in g)
        out.append({**dict(zip(keys, key)), "warianty": len(g), "tp": tp, "fp": pr - tp, "fn": ex - tp,
                    "predykcje": pr, "oczekiwane": ex, "precyzja": p, "czulosc": rc, "f1": f1_score(p, rc),
                    "makro_f1": mean(r["f1"] for r in g),
                    "zgodnosc_typu": ratio(sum(r["zgodne_typ"] for r in g), tp),
                    "zgodnosc_etykiety": ratio(sum(r["zgodne_etykieta"] for r in g), tp),
                    "zgodnosc_opcji": ratio(sum(r["zgodne_opcje"] for r in g), tp),
                    "scisla_zgodnosc_tresci": ratio(strict, tp),
                    "scisla_f1": f1_score(ratio(strict, pr), ratio(strict, ex))})
    return out
SHORT_DOC = {"Nadpłata podatku": "Nadpłata", "Numer porządkowy nieruchomości": "Nr porządkowy"}
def variant_label(r):
    return f"{SHORT_DOC.get(r['dokument'], r['dokument'])} · {r['format'].upper()}"
EVAL_COLUMNS = ["dokument", "format", "reprezentacja", "zakres", "predykcje", "oczekiwane", "tp", "fp", "fn",
                "precyzja", "czulosc", "f1", "zgodne_typ", "zgodne_etykieta", "zgodne_opcje", "zgodne_scisla"]
table("trafnosc_warianty", sorted(EVAL, key=lambda r: (r["dokument"], r["reprezentacja"], r["zakres"], r["format"])),
      limit=80, columns=EVAL_COLUMNS, title="Trafność każdego wariantu względem wzorca")
table("trafnosc_zbiorczo", aggregate(EVAL, ["reprezentacja", "zakres"]), title="Trafność zbiorcza: reprezentacja × zakres")
table("trafnosc_dokument", aggregate([r for r in EVAL if r["rep"] != "simple"], ["dokument", "reprezentacja", "zakres"]),
      title="Trafność według dokumentu")
form_content = sorted([r for r in EVAL if r["rep"] == "form" and r["zakres"] == CONTENT], key=lambda r: (r["dokument"], r["format"]))
form_common = sorted([r for r in EVAL if r["rep"] == "form" and r["zakres"] == COMMON], key=lambda r: (r["dokument"], r["format"]))
form_all = sorted([r for r in EVAL if r["rep"] == "form" and r["zakres"] == ALL_INPUTS], key=lambda r: (r["dokument"], r["format"]))
bar_chart("trafnosc_tresc_form", "Treść wniosku — form: precyzja, czułość i F1 (dopasowanie automatyczne)",
          [variant_label(r) for r in form_content],
          [("Precyzja", [r["precyzja"] for r in form_content]), ("Czułość", [r["czulosc"] for r in form_content]),
           ("F1", [r["f1"] for r in form_content])],
          note="Główny miernik: pola merytoryczne wniosku. Dopasowanie automatyczne, próg 0,60.")
bar_chart("trafnosc_wspolny_szablon", "Wspólny szablon — form: F1 oraz zgodność ścisła (wszystkie pola wejściowe)",
          [variant_label(r) for r in form_common],
          [("F1 wspólnego szablonu", [r["f1"] for r in form_common]),
           ("F1 wszystkich pól wejściowych", [r["f1"] for r in form_all])],
          note="Wysoka zgodność szablonu odzwierciedla generator formularza, a nie jakość odczytu dokumentu.")
pooled = aggregate(EVAL, ["reprezentacja", "zakres"])
bar_chart("trafnosc_zbiorcza", "Trafność zbiorcza według reprezentacji i zakresu",
          [f"{r['reprezentacja']} · {r['zakres']}" for r in pooled],
          [("Precyzja", [r["precyzja"] for r in pooled]), ("Czułość", [r["czulosc"] for r in pooled]),
           ("F1", [r["f1"] for r in pooled])])

### Które pola wzorca zostały znalezione w którym formacie?
Heatmap pokazuje, czy pole **treści wniosku** zostało znalezione w danym formacie:
**A** — dopasowanie automatyczne NLP, **R** — dopasowanie po ręcznej weryfikacji
semantycznej, **—** — brak. To szybki
sposób na upewnienie się, ze analiza pól odzwierciedla rzeczywistą użyteczność rozwiązania.

In [ ]:
def detection_status(source_id, rep, ref_index):
    pred, exp, matches, extra, missing = MATCH_STORE[(source_id, rep, CONTENT)]
    if any(j == ref_index for _, j, _, _ in matches):
        return "A", 1.0
    _, _, pairs = MATCH_STORE[(source_id, rep, "ręczne")]
    for _, j, tier in pairs:
        if j == ref_index:
            return ("R" if tier == "pewne" else "R?"), 0.5
    return "—", 0.0
for ref_key in references:
    sources = [sid for sid, m in sorted(source_meta.items()) if m["ref_key"] == ref_key]
    ref_content = [r for r in reference_rows[ref_key] if r["is_input"] and r["scope"] == CONTENT]
    for rep, rep_label in (("form", "form"), ("fields", "fields")):
        grid = [[detection_status(sid, rep, j) for sid in sources] for j in range(len(ref_content))]
        heatmap(f"mapa_{rep}_{DOC_SHORT[ref_key][:6].strip()}", f"Wykrycie pól wzorca — {DOC_SHORT[ref_key]} ({rep_label})",
                [r["label"] for r in ref_content], [source_meta[s]["format"].upper() for s in sources],
                [[v for _, v in row] for row in grid], texts=[[t for t, _ in row] for row in grid], scale=1.0,
                note="A = dopasowanie automatyczne; R = tylko ręczny odpowiednik; R? = ręczny, niepewny; — = brak")

### Dopasowanie automatyczne a ręczna weryfikacja semantyczna
Dopasowanie leksykalne zaniża trafność, gdy ta sama informacja ma inne słowa
(np. „Sposób rozliczenia nadpłaty” i „Wybierz, co mamy zrobić z nadpłatą”). Tabela
odpowiedników jest jawna (zmienna `MANUAL` w rozdziale 4), więc można ją zmienić i
przeliczyć. „Pewne” — odpowiedniki jednoznaczne; „niepewne” — wymagają decyzji
merytorycznej (np. *Inne załączniki* ↔ *Plik deklaracji podatkowej*).

In [ ]:
auto_rows = [{**r, "dopasowanie": "automatyczne NLP"} for r in EVAL if r["zakres"] == CONTENT and r["rep"] in ("form", "fields")]
compare = []
for r in auto_rows + MANUAL_ROWS:
    if r["rep"] in ("form", "fields"):
        compare.append({"dokument": r["dokument"], "format": r["format"], "reprezentacja": r["reprezentacja"], "dopasowanie": r["dopasowanie"],
                        "tp": r["tp"], "predykcje": r["predykcje"], "oczekiwane": r["oczekiwane"],
                        "precyzja": r["precyzja"], "czulosc": r["czulosc"], "f1": r["f1"], "rep": r["rep"], "zakres": CONTENT})
table("automatyczne_vs_reczne", sorted(compare, key=lambda r: (r["dokument"], r["reprezentacja"], r["format"], r["dopasowanie"])),
      limit=90, columns=["dokument", "format", "reprezentacja", "dopasowanie", "tp", "predykcje", "oczekiwane", "precyzja", "czulosc", "f1"],
      title="Treść wniosku: dopasowanie automatyczne i ręczne (wszystkie warianty)")
modes = ["automatyczne NLP", "ręczne: pewne", "ręczne: pewne+niepewne"]
pooled_modes = []
for rep in ("form", "fields"):
    for doc in sorted({r["dokument"] for r in compare}):
        for mode in modes:
            g = [r for r in compare if r["rep"] == rep and r["dokument"] == doc and r["dopasowanie"] == mode]
            tp, pr, ex = sum(r["tp"] for r in g), sum(r["predykcje"] for r in g), sum(r["oczekiwane"] for r in g)
            pooled_modes.append({"reprezentacja": REP_NAMES[rep], "dokument": doc, "dopasowanie": mode, "tp": tp,
                                 "predykcje": pr, "oczekiwane": ex, "precyzja": ratio(tp, pr), "czulosc": ratio(tp, ex),
                                 "f1": f1_score(ratio(tp, pr), ratio(tp, ex))})
table("automatyczne_vs_reczne_zbiorczo", pooled_modes, title="Treść wniosku: porównanie metod dopasowania (zbiorczo)")
for rep in ("form", "fields"):
    chosen = [r for r in pooled_modes if r["reprezentacja"] == REP_NAMES[rep]]
    docs = sorted({r["dokument"] for r in chosen})
    bar_chart(f"dopasowanie_{rep}", f"Treść wniosku ({rep}): F1 według sposobu dopasowania",
              docs, [(mode, [next(r["f1"] for r in chosen if r["dokument"] == d and r["dopasowanie"] == mode) for d in docs])
                     for mode in modes],
              note="Ręczna weryfikacja to propozycja analityczna; różnica pokazuje zakres niedoszacowania przez NLP leksykalne.")

### Atrybuty dopasowanych pól, tytuł formularza i reguły walidacji
* **Tytuł** (`title`) — porównanie z tytułem wzorca. 
* **Rozbieżności atrybutów** — dopasowane pola, w których różni się etykieta, typ, opcje,
  tryb tylko do odczytu, wartość domyślna lub warunki stanów.
* **Walidacje** — wzorce ich nie zawierają, więc zgodności nie oceniamy; raportujemy tylko,
  ile reguł ekstrakcja dodała.

In [ ]:
table("tytul_formularza", [{"dokument": q["dokument"], "format": q["format"], "przetwarzanie_tekstu": q["przetwarzanie_tekstu"],
                            "tytul_form": q["tytul_form"], "tytul_form_zgodny": q["tytul_form_zgodny"],
                            "tytul_fields": q["tytul_fields"], "tytul_fields_zgodny": q["tytul_fields_zgodny"]}
                           for q in quality], limit=14, title="Zgodność tytułu formularza z wzorcem")
mismatches = []
for r in AUDIT:
    if r["rep"] != "form" or r["zakres"] != ALL_INPUTS:
        continue
    bad = [k for k in ("etykieta", "typ", "opcje", "tryb_odczytu", "wartosc_domyslna", "stany") if not r["zgodne_" + k]]
    if bad:
        mismatches.append({"dokument": r["dokument"], "format": r["format"], "etykieta_wyniku": r["etykieta_wyniku"],
                           "etykieta_wzorca": r["etykieta_wzorca"], "sekcja_wzorca": r["sekcja_wzorca"],
                           "rozbieżne_atrybuty": ", ".join(bad), "podobienstwo": r["podobienstwo"]})
mismatch_summary = Counter((r["dokument"], r["etykieta_wyniku"], r["etykieta_wzorca"], r["rozbieżne_atrybuty"]) for r in mismatches)
table("rozbieznosci_atrybutow", [{"dokument": k[0], "etykieta_wyniku": k[1], "etykieta_wzorca": k[2], "rozbieżne_atrybuty": k[3],
                                  "liczba_wariantow": v} for k, v in mismatch_summary.most_common()], limit=40,
      title="Dopasowane pola z rozbieżnymi atrybutami (form, zagregowane po wariantach)")
validation_rows = []
SOURCE_OF = {(m["doc_short"], m["format"]): sid for sid, m in source_meta.items()}
for q_ in quality:
    rows = per_record[by_source[SOURCE_OF[(q_["dokument"], q_["format"])]]["form"]["name"]]
    inputs = [r for r in rows if r["is_input"]]
    validation_rows.append({"dokument": q_["dokument"], "format": q_["format"], "pola_wejsciowe": len(inputs),
                            "pola_z_regulami_walidacji": sum(r["rules"] != "[]" for r in inputs),
                            "pola_REQUIRED": sum(r["required"] for r in inputs),
                            "reguly_walidacji_we_wzorcu": 0})
table("walidacje_nadwyzka", validation_rows, limit=14, title="Reguły walidacji w naszych formularzach (wzorzec: brak)")

## 6. Pytanie 2 — trafność według kategorii pól
Kategorie są **heurystyczne** (słowa kluczowe i typ pola), więc ich granice można
zakwestionować; ich zastosowanie jest identyczne dla wzorca i wyniku. Dla każdej
kategorii podajemy **czułość** (jaka część pól wzorca tej kategorii została znaleziona)
i **precyzję** (jaka część naszych pól tej kategorii ma odpowiednik we wzorcu). Statystyki
pochodzą z jednego globalnego dopasowania jeden do jednego — nie z korzystniejszego
dopasowania osobno w każdej kategorii.

In [ ]:
DIM_NAMES = {"category": "kategoria", "form_type": "typ_pola"}
def category_stats(rep, scope, dimension):
    stats = defaultdict(lambda: {"oczekiwane": 0, "znalezione": 0, "predykcje": 0, "trafne": 0})
    for (sid, r, sc), value in MATCH_STORE.items():
        if r != rep or sc != scope:
            continue
        pred, exp, matches, _, _ = value
        matched_p, matched_e = {i for i, _, _, _ in matches}, {j for _, j, _, _ in matches}
        for j, row in enumerate(exp):
            stats[row[dimension]]["oczekiwane"] += 1
            stats[row[dimension]]["znalezione"] += j in matched_e
        for i, row in enumerate(pred):
            stats[row[dimension]]["predykcje"] += 1
            stats[row[dimension]]["trafne"] += i in matched_p
    out = []
    for key, v in sorted(stats.items()):
        p, rc = ratio(v["trafne"], v["predykcje"]), ratio(v["znalezione"], v["oczekiwane"])
        out.append({"reprezentacja": REP_NAMES[rep], "zakres": scope, DIM_NAMES[dimension]: key, **v,
                    "precyzja": p, "czulosc": rc, "f1": f1_score(p, rc)})
    return out
cat_all = category_stats("form", ALL_INPUTS, "category")
cat_common = category_stats("form", COMMON, "category")
cat_content = category_stats("form", CONTENT, "category")
cat_raw = category_stats("fields", CONTENT, "category")
for name, rows_, title in (("kategorie_form_wszystkie", cat_all, "Kategorie — form, wszystkie pola wejściowe"),
                           ("kategorie_form_tresc", cat_content, "Kategorie — form, treść wniosku"),
                           ("kategorie_form_szablon", cat_common, "Kategorie — form, wspólny szablon"),
                           ("kategorie_fields_tresc", cat_raw, "Kategorie — fields, treść wniosku")):
    table(name, [{**r} for r in rows_], title=title)
table("typy_pol_form_wszystkie", category_stats("form", ALL_INPUTS, "form_type"), title="Typy pól FormDefinition — form, wszystkie pola")
table("typy_pol_form_tresc", category_stats("form", CONTENT, "form_type"), title="Typy pól FormDefinition — form, treść wniosku")
cats_ = [r for r in cat_all if r["oczekiwane"] > 0]
bar_chart("kategorie_czulosc_precyzja", "Kategorie pól (form, wszystkie pola): czułość i precyzja",
          [f"{r['kategoria']} (n={r['oczekiwane']})" for r in cats_],
          [("Czułość", [r["czulosc"] for r in cats_]), ("Precyzja", [r["precyzja"] for r in cats_])],
          note="n = liczba pól wzorca w kategorii, zsumowana po 14 wariantach.")
cats_c = [r for r in cat_content if r["oczekiwane"] > 0]
bar_chart("kategorie_tresc", "Kategorie pól — tylko treść wniosku (form)",
          [f"{r['kategoria']} (n={r['oczekiwane']})" for r in cats_c],
          [("Czułość", [r["czulosc"] for r in cats_c]), ("Precyzja", [r["precyzja"] for r in cats_c])],
          note="Małe liczebności: po 7 wariantów na dokument; traktować jako obserwację, nie estymator ogólny.")
by_doc_cat = []
for ref_key in references:
    for sc in (CONTENT,):
        stats = defaultdict(lambda: [0, 0])
        for (sid, r, s), v in MATCH_STORE.items():
            if r == "form" and s == sc and source_meta[sid]["ref_key"] == ref_key:
                _, exp, matches, _, _ = v
                found = {j for _, j, _, _ in matches}
                for j, row in enumerate(exp):
                    stats[row["category"]][0] += 1
                    stats[row["category"]][1] += j in found
        for key, (n, f) in sorted(stats.items()):
            by_doc_cat.append({"dokument": DOC_SHORT[ref_key], "zakres": sc, "kategoria": key, "oczekiwane": n,
                               "znalezione": f, "czulosc": ratio(f, n)})
table("kategorie_wg_dokumentu", by_doc_cat, title="Czułość według kategorii i dokumentu (treść wniosku, form)")

## 7. Pytanie 3 — trafność według metody ekstrakcji
Rozróżniamy trzy rzeczy, które **nie są niezależnymi eksperymentami**:
1. **Format wejściowy** (DOC/DOCX/ODS/XLSX/PDF vs JPG/JPEG/PNG).
2. **Sposób pozyskania tekstu** (`text_source`): *tekst z PDF + VLM* albo *OCR (Tesseract) + VLM*.
   W tym zbiorze jest on w pełni powiązany z formatem: formaty biurowe i PDF idą
   ścieżką PDF, obrazy ścieżką OCR. Nie da się więc rozdzielić wpływu formatu od
   wpływu OCR.
3. **Detektor pola** (`detector`): `vlm` lub `dots+vlm` — pole potwierdzone dodatkowo
   detekcją kropek. To pochodzenie złożone, a nie dwa niezależne detektory.
Reprezentacje `fields`, `simple` i `form` to ta sama ekstrakcja w różnych postaciach.
Każda ścieżka ma od 2 do 5 wariantów na dokument (nadpłata: 4 PDF / 3 OCR; nieruchomość:
5 PDF / 2 OCR), więc różnice trzeba czytać ostrożnie i **tylko w obrębie jednego dokumentu**.

In [ ]:
pipeline_rows = aggregate([r for r in EVAL if r["rep"] in ("form", "fields")],
                          ["dokument", "reprezentacja", "przetwarzanie_tekstu", "zakres"])
table("metoda_przetwarzanie_tekstu", pipeline_rows, title="Trafność według sposobu pozyskania tekstu (osobno dla dokumentów)")
table("metoda_przetwarzanie_zbiorczo", aggregate([r for r in EVAL if r["rep"] in ("form", "fields")],
                                                  ["reprezentacja", "przetwarzanie_tekstu", "zakres"]),
      title="Trafność według sposobu pozyskania tekstu (zbiorczo, obie reprezentacje)")
table("metoda_format", aggregate([r for r in EVAL if r["rep"] == "form"], ["dokument", "format", "zakres"]), limit=50,
      title="Trafność według formatu wejściowego (form)")
pr_content = [r for r in pipeline_rows if r["zakres"] == CONTENT and r["reprezentacja"] == REP_NAMES["form"]]
bar_chart("metoda_pipeline_tresc", "Treść wniosku (form): F1 według sposobu pozyskania tekstu",
          [f"{r['dokument']} · {r['przetwarzanie_tekstu']} (n={r['warianty']})" for r in pr_content],
          [("Precyzja", [r["precyzja"] for r in pr_content]), ("Czułość", [r["czulosc"] for r in pr_content]),
           ("F1", [r["f1"] for r in pr_content])],
          note="Powiązanie z formatem: ścieżka PDF = DOC/DOCX/ODS/XLSX/PDF; OCR = JPEG/JPG/PNG.")
pr_common = [r for r in pipeline_rows if r["zakres"] == COMMON and r["reprezentacja"] == REP_NAMES["form"]]
bar_chart("metoda_pipeline_szablon", "Wspólny szablon (form): F1 według sposobu pozyskania tekstu",
          [f"{r['dokument']} · {r['przetwarzanie_tekstu']} (n={r['warianty']})" for r in pr_common],
          [("F1", [r["f1"] for r in pr_common])])

### Detektory pól i deklarowana pewność (`fields`)
Pola surowej ekstrakcji obejmują także tożsamość, kontakt, datę i podpis. Do oceny
trafności używamy zakresu **treści wniosku**. „Trafne (auto)” — pole ma odpowiednik w
dopasowaniu automatycznym; „trafne (ręcznie pewne)” — w ręcznej tabeli odpowiedników.
Brak dopasowania nie dowodzi błędu: może oznaczać pole spoza wzorca (np. podpis).
**Uwaga:** detektor jest silnie powiązany z dokumentem (we wniosku o numer nieruchomości
dominuje `dots+vlm`), więc porównanie detektorów należy czytać razem z tabelą
„według dokumentu”; różnice mogą wynikać z dokumentu, a nie z detektora.
Pewność (`confidence`) jest wartością zadeklarowaną przez ekstraktor, nie skalibrowanym
prawdopodobieństwem.

In [ ]:
detector_rows = []
for sid, reps in sorted(by_source.items()):
    m = source_meta[sid]
    raw = [r for r in per_record[reps["fields"]["name"]]]
    pred, exp, matches, _, _ = MATCH_STORE[(sid, "fields", CONTENT)]
    pred_ids = {r["uid"]: i for i, r in enumerate(pred)}
    auto = {i for i, _, _, _ in matches}
    _, _, pairs = MATCH_STORE[(sid, "fields", "ręczne")]
    manual_sure = {i for i, _, t in pairs if t == "pewne"}
    for r in raw:
        i = pred_ids.get(r["uid"])
        detector_rows.append({"dokument": m["doc_short"], "format": m["format"], "przetwarzanie_tekstu": m["pipeline"],
                              "detektor": r["detector"], "rola": r["role"], "kategoria": r["category"],
                              "pewnosc": r["confidence"], "w_tresci": i is not None,
                              "auto": i in auto if i is not None else None,
                              "reczne_pewne": i in manual_sure if i is not None else None})
def detector_table(keys):
    groups = defaultdict(list)
    for r in detector_rows:
        groups[tuple(r[k] for k in keys)].append(r)
    out = []
    for key, g in sorted(groups.items(), key=lambda x: str(x[0])):
        content = [r for r in g if r["w_tresci"]]
        out.append({**dict(zip(keys, key)), "pola_ogolem": len(g), "pola_w_tresci_wniosku": len(content),
                    "srednia_pewnosc": mean(r["pewnosc"] for r in g),
                    "trafne_auto": sum(bool(r["auto"]) for r in content),
                    "trafne_reczne_pewne": sum(bool(r["reczne_pewne"]) for r in content),
                    "udzial_auto": ratio(sum(bool(r["auto"]) for r in content), len(content)),
                    "udzial_reczny": ratio(sum(bool(r["reczne_pewne"]) for r in content), len(content))})
    return out
det = detector_table(["detektor"])
table("detektory", det, title="Detektory pól (fields, 14 wariantów)")
table("detektory_i_przetwarzanie", detector_table(["przetwarzanie_tekstu", "detektor"]), title="Detektory według sposobu pozyskania tekstu")
table("detektory_i_dokument", detector_table(["dokument", "detektor"]), title="Detektory według dokumentu (kontrola czynnika zakłócającego)")
table("detektory_kategorie", detector_table(["detektor", "kategoria"]), limit=40, title="Detektory według kategorii pól")
bar_chart("detektory_pewnosc_trafnosc", "Detektory: pewność deklarowana a udział pól zgodnych z wzorcem (treść wniosku)",
          [f"{r['detektor']} (n={r['pola_w_tresci_wniosku']})" for r in det if r["pola_w_tresci_wniosku"]],
          [("Średnia pewność", [r["srednia_pewnosc"] for r in det if r["pola_w_tresci_wniosku"]]),
           ("Udział zgodnych (auto)", [r["udzial_auto"] for r in det if r["pola_w_tresci_wniosku"]]),
           ("Udział zgodnych (ręczne pewne)", [r["udzial_reczny"] for r in det if r["pola_w_tresci_wniosku"]])],
          note="Pewność jest deklaracją ekstraktora; ustalono ją niezależnie od wzorca i nie jest skalibrowana.")
bins = defaultdict(list)
for r in detector_rows:
    if r["w_tresci"] and r["pewnosc"] is not None:
        bins[f"{r['pewnosc']:.2f}"].append(r)
table("pewnosc_a_zgodnosc", [{"deklarowana_pewnosc": k, "pola": len(v), "trafne_auto": sum(bool(x["auto"]) for x in v),
                              "trafne_reczne_pewne": sum(bool(x["reczne_pewne"]) for x in v),
                              "udzial_reczny": ratio(sum(bool(x["reczne_pewne"]) for x in v), len(v))}
                             for k, v in sorted(bins.items())], title="Deklarowana pewność a zgodność pól z wzorcem")
rep_check = []
for sid, reps in sorted(by_source.items()):
    a = next(r for r in EVAL if r["source_id"] == sid and r["rep"] == "fields")
    b = next(r for r in EVAL if r["source_id"] == sid and r["rep"] == "simple")
    rep_check.append({"dokument": source_meta[sid]["doc_short"], "format": source_meta[sid]["format"],
                      "tp_fields": a["tp"], "tp_simple": b["tp"], "fp_fields": a["fp"], "fp_simple": b["fp"],
                      "fn_fields": a["fn"], "fn_simple": b["fn"], "f1_fields": a["f1"], "f1_simple": b["f1"],
                      "identyczne": (a["tp"], a["fp"], a["fn"]) == (b["tp"], b["fp"], b["fn"])})
table("fields_vs_simple", rep_check, limit=14, title="Porównanie reprezentacji fields i simple")
print("fields i simple dają identyczne TP/FP/FN w", sum(r["identyczne"] for r in rep_check), "z", len(rep_check), "wariantów.")

## 8. Diagnostyka NLP
Sprawdzamy jakość samego dopasowania: rozkład podobieństwa przyjętych par, pary
graniczne do ręcznej weryfikacji, najlepsze kandydatki dla pól wzorca, których nie
udało się dopasować, wrażliwość na próg i słownictwo dokumentów.

In [ ]:
table("podobienstwo_wg_podstawy", [{"podstawa": k[0], "zakres": k[1], "liczba": len(v),
                                    "srednie_podobienstwo": mean(x["podobienstwo"] for x in v),
                                    "minimalne_podobienstwo": min(x["podobienstwo"] for x in v)}
                                   for k, v in sorted(defaultdict(list, {
                                       key: [a for a in AUDIT if (a["podstawa"], a["zakres"]) == key]
                                       for key in {(a["podstawa"], a["zakres"]) for a in AUDIT}}).items())],
      title="Przyjęte dopasowania według podstawy i zakresu")
borderline = [a for a in AUDIT if a["podstawa"] == "podobieństwo NLP" and a["podobienstwo"] < 0.85]
border_summary = Counter((a["dokument"], a["reprezentacja"], a["zakres"], a["etykieta_wyniku"], a["etykieta_wzorca"]) 
for a in borderline)
table("pary_graniczne", [{"dokument": k[0], "reprezentacja": k[1], "zakres": k[2], "etykieta_wyniku": k[3],
                          "etykieta_wzorca": k[4], "liczba_wariantow": v,
                          "srednie_podobienstwo": mean(a["podobienstwo"] for a in borderline
                                                       if (a["dokument"], a["reprezentacja"], a["zakres"], a["etykieta_wyniku"], a["etykieta_wzorca"]) == k)}
                         for k, v in border_summary.most_common()], limit=40,
      title="Dopasowania NLP o podobieństwie < 0,85 — do ręcznej weryfikacji")
candidates = defaultdict(lambda: {"max": 0.0, "warianty": set()})
for (sid, rep, scope), value in MATCH_STORE.items():
    if scope not in (CONTENT, COMMON) or rep not in ("form", "fields") or (rep == "fields" and scope != CONTENT):
        continue
    pred, exp, matches, extra, missing = value
    for j in missing:
        scored = sorted(((similarity(pred[i], exp[j]), i) for i in extra), reverse=True)[:3]
        for score, i in scored:
            key = (source_meta[sid]["doc_short"], REP_NAMES[rep], scope, exp[j]["label"], pred[i]["label"])
            candidates[key]["max"] = max(candidates[key]["max"], score)
            candidates[key]["warianty"].add(source_meta[sid]["format"])
table("kandydaci_dla_brakujacych", [{"dokument": k[0], "reprezentacja": k[1], "zakres": k[2], "pole_wzorca_bez_dopasowania": k[3],
                                    "najlepszy_niedopasowany_kandydat": k[4], "maks_podobienstwo": v["max"], "liczba_wariantow": len(v["warianty"])}
                                   for k, v in sorted(candidates.items(), key=lambda x: (-x[1]["max"], x[0]))], limit=40,
      title="Najlepsze kandydatki (spośród niedopasowanych pól wyniku) dla pól wzorca bez dopasowania")
sens_rows = []
for rep in ("form", "fields"):
    for threshold in THRESHOLDS:
        g = [r for r in SENS if r["rep"] == rep and r["prog"] == threshold]
        tp, pr, ex = sum(r["tp"] for r in g), sum(r["predykcje"] for r in g), sum(r["oczekiwane"] for r in g)
        sens_rows.append({"reprezentacja": REP_NAMES[rep], "prog": threshold, "tp": tp, "predykcje": pr, "oczekiwane": ex,
                          "precyzja": ratio(tp, pr), "czulosc": ratio(tp, ex), "f1": f1_score(ratio(tp, pr), ratio(tp, ex))})
table("wrazliwosc_na_prog", sens_rows, title="Wrażliwość wyniku na próg podobieństwa (treść wniosku, zbiorczo)")
bar_chart("wrazliwosc_na_prog", "Wrażliwość F1 na próg podobieństwa (treść wniosku)",
          [f"próg {t:.2f}" for t in THRESHOLDS],
          [(REP_NAMES[rep], [next(r["f1"] for r in sens_rows if r["reprezentacja"] == REP_NAMES[rep] and r["prog"] == t)
                             for t in THRESHOLDS]) for rep in ("form", "fields")],
          note="Próg 0,60 ustalono z góry; wykres pokazuje niepewność, nie służy do wyboru progu.")
doc_terms = defaultdict(Counter)
for r in all_rows + [x for v in reference_rows.values() for x in v]:
    if r["is_input"]:
        doc_terms[r["document"]].update(set(stems(r["label"])))
term_df = Counter(t for c in doc_terms.values() for t in c)
table("slownictwo_dokumentow", [{"dokument": d, "wyroznione_rdzenie": [t for t, _ in sorted(
    ((t, (1 + math.log(n)) * (1 + math.log((1 + len(doc_terms)) / (1 + term_df[t])))) for t, n in c.items()),
    key=lambda x: -x[1])[:12]]} for d, c in sorted(doc_terms.items())], title="Wyróżniające rdzenie słownictwa (TF-IDF) w dokumentach")
lexical = []
for rec in records:
    if rec["representation"] == "simple":
        continue
    rows = [r for r in per_record[rec["name"]] if r["is_input"]]
    words = [w for r in rows for w in norm(r["label"]).split()]
    lexical.append({"dokument": source_meta[rec["source_id"]]["doc_short"], "format": source_meta[rec["source_id"]]["format"],
                    "reprezentacja": rec["representation"], "pola": len(rows), "tokeny": len(words),
                    "unikatowe_tokeny": len(set(words)), "udzial_unikatowych": ratio(len(set(words)), len(words)),
                    "srednia_dlugosc_etykiety": mean(len(r["label"]) for r in rows)})
table("statystyki_leksykalne", lexical, limit=30, title="Statystyki leksykalne etykiet według wariantu")

## 9. Spójność ekstrakcji między formatami
Porównanie surowych pól (`fields`) tego samego dokumentu w różnych formatach, bez
użycia wzorca. F1 = 2 · dopasowane / (pola A + pola B). To **spójność, nie trafność**:
dwa formaty mogą zgodnie się mylić, a zgodność nie wskazuje, który jest poprawny.

In [ ]:
format_pairs = []
for ref_key in references:
    sources = [sid for sid, m in sorted(source_meta.items()) if m["ref_key"] == ref_key]
    labels = [source_meta[s]["format"].upper() for s in sources]
    matrix = [[1.0] * len(sources) for _ in sources]
    for (ia, sa), (ib, sb) in itertools.combinations(list(enumerate(sources)), 2):
        a, b = per_record[by_source[sa]["fields"]["name"]], per_record[by_source[sb]["fields"]["name"]]
        res, matches, _, _ = evaluate(a, b, scoped=False)
        matrix[ia][ib] = matrix[ib][ia] = res["f1"]
        format_pairs.append({"dokument": DOC_SHORT[ref_key], "format_A": labels[ia], "format_B": labels[ib],
                             "pola_A": len(a), "pola_B": len(b), "dopasowane": len(matches), "f1_spojnosci": res["f1"],
                             "zgodnosc_etykiet": ratio(res["zgodne_etykieta"], len(matches)),
                             "zgodnosc_typow": ratio(res["zgodne_typ"], len(matches))})
    heatmap(f"spojnosc_{DOC_SHORT[ref_key][:6].strip()}", f"Spójność surowych pól między formatami — {DOC_SHORT[ref_key]}",
            labels, labels, matrix, scale=1.0, note="F1 dopasowania pól dwóch formatów; to nie jest trafność względem wzorca.")
table("spojnosc_par_formatow", format_pairs, limit=45, title="Spójność par formatów (fields)")
stability = []
for ref_key in references:
    sources = [sid for sid, m in sorted(source_meta.items()) if m["ref_key"] == ref_key]
    presence = defaultdict(set)
    for sid in sources:
        for r in per_record[by_source[sid]["fields"]["name"]]:
            presence[norm(r["label"])].add(source_meta[sid]["format"])
    original = {norm(r["label"]): r["label"] for sid in sources for r in per_record[by_source[sid]["fields"]["name"]]}
    for label, formats in sorted(presence.items(), key=lambda x: (len(x[1]), x[0])):
        if len(formats) < len(sources):
            stability.append({"dokument": DOC_SHORT[ref_key], "etykieta": original[label], "formaty_z_etykieta": len(formats),
                              "z_formatow": len(sources), "obecna_w": sorted(formats),
                              "brak_w": sorted({source_meta[s]["format"] for s in sources} - formats)})
table("niestabilne_etykiety", stability, limit=40, title="Etykiety obecne nie we wszystkich formatach tego samego dokumentu")